Libraries Imported

In [22]:
from pathlib import Path
from datetime import timedelta
from scipy.spatial.distance import cdist
import pandas as pd
import geopandas as gpd
import numpy as np

Files, Office Location, Coordinate Reference system, current date and number of officers loaded

In [21]:
#Synthetic Data File and Road Map
file = "supplyFakeData.xlsx"
roadMap = "borderRoadNetwork.gpkg"
#Coordinate reference system: Ordnance Survey Map
coordinateRefStyle = "EPSG:27700"
#Location of SBC Headquarters where pool cars and sampling equipment located
officeCoordinates = (357642, 631785)
#Project end date used as placeholder for a random schedule run in 2027
currentDate = pd.Timestamp("2027-07-01")
#number of officers to conduct sampling
samplers = 4

Sampling time data and travel time data loaded

In [3]:
startTime = 9 * 60 + 30
endTime = 14 * 60
samplingTime = endTime - startTime
averageMPH = 40
sampleDuration = 45
maxDailySupplies = 3
metresPerMin = averageMPH * 1609 / 60

Synthetic data read and loaded into a geodataframe with Eastings and Northings as location information

In [4]:
supplyDf = pd.read_excel(file)
supplyDf["lastSampleDate"] = pd.to_datetime(supplyDf["lastSampleDate"])
supplies = gpd.GeoDataFrame(supplyDf, geometry = gpd.points_from_xy(supplyDf["supplyEastings"],supplyDf["supplyNorthings"]), crs = coordinateRefStyle)

Road map data loaded

In [5]:
roadNetwork = gpd.read_file(roadMap, layer = "road_link")
roadNodes = gpd.read_file(roadMap, layer = "road_node")

Sampling Window and Risk Classification Adjustment

In [6]:
earliestMonths = 9
latestMonths = 15

riskClassPriority = {
    1: 75,
    2: 50,
    3: 40,
    4: 20,
    5: 5,
    6: 0
}

In [7]:
samplingYear = 2027

newYearStart = pd.Series(pd.Timestamp(f"{samplingYear}-01-01"), index = supplyDf.index)

newYearEnd = pd.Series(pd.Timestamp(f"{samplingYear}-12-31"), index = supplyDf.index)


nineMonths = supplyDf["lastSampleDate"] + pd.DateOffset(months=earliestMonths)

fifteenMonths = supplyDf["lastSampleDate"] + pd.DateOffset(months=latestMonths)

supplyDf ["earliestDate"] = pd.concat (
    [nineMonths, newYearStart],
    axis = 1
).max(axis = 1)

supplyDf ["latestDate"] = pd.concat (
    [fifteenMonths, newYearEnd],
    axis = 1
).min(axis = 1)

supplyDf ["latestDate"] = supplyDf["latestDate"].where(supplyDf["latestDate"] >= supplyDf["earliestDate"], supplyDf ["earliestDate"])

display(supplyDf[["supplyName", "lastSampleDate", "earliestDate", "latestDate"]].head(10))


,supplyName,lastSampleDate,earliestDate,latestDate
0,A,2025-11-01,2027-01-01,2027-02-01
1,B,2026-02-02,2027-01-01,2027-05-02
2,C,2026-06-03,2027-03-03,2027-09-03
3,D,2026-08-04,2027-05-04,2027-11-04
4,E,2026-02-01,2027-01-01,2027-05-01
5,F,2026-02-02,2027-01-01,2027-05-02
6,G,2025-03-04,2027-01-01,2027-01-01
7,H,2026-05-28,2027-02-28,2027-08-28
8,I,2026-05-05,2027-02-05,2027-08-05
9,J,2026-07-05,2027-04-05,2027-10-05


In [36]:
supplyDf["daysTilDue"] = (supplyDf["latestDate"] - currentDate).dt.days
supplyDf["riskAddition"] = supplyDf["supplyRisk"].map(riskClassPriority)
supplyDf["effectiveDeadline"] = supplyDf["latestDate"] - pd.to_timedelta(supplyDf["riskAddition"], unit = "D")
supplyDf["daysTilDuePriority"] = (supplyDf["effectiveDeadline"] - currentDate).dt.days
display(supplyDf[["supplyName", "supplyRisk", "latestDate", "daysTilDue", "riskAddition", "effectiveDeadline", "daysTilDuePriority"]].head(10))

,supplyName,supplyRisk,latestDate,daysTilDue,riskAddition,effectiveDeadline,daysTilDuePriority
0,A,1,2027-02-01,-150,75,2026-11-18,-225
1,B,2,2027-05-02,-60,50,2027-03-13,-110
2,C,3,2027-09-03,64,40,2027-07-25,24
3,D,4,2027-11-04,126,20,2027-10-15,106
4,E,3,2027-05-01,-61,40,2027-03-22,-101
5,F,2,2027-05-02,-60,50,2027-03-13,-110
6,G,1,2027-01-01,-181,75,2026-10-18,-256
7,H,5,2027-08-28,58,5,2027-08-23,53
8,I,6,2027-08-05,35,0,2027-08-05,35
9,J,6,2027-10-05,96,0,2027-10-05,96


In [12]:
supplyDf["samplingStatus"] = "Not Yet Due"

supplyDf.loc[supplyDf["latestDate"] < currentDate, "samplingStatus"] = "Overdue"
supplyDf.loc[(supplyDf["earliestDate"] <= currentDate) & (supplyDf["latestDate"] >= currentDate), "samplingStatus"] = "Sample Now"

print(supplyDf["samplingStatus"].value_counts())
display(supplyDf[["supplyName", "lastSampleDate", "earliestDate", "latestDate", "samplingStatus"]].head(10))

samplingStatus
Sample Now     14
Overdue        10
Not Yet Due     2
Name: count, dtype: int64


,supplyName,lastSampleDate,earliestDate,latestDate,samplingStatus
0,A,2025-11-01,2027-01-01,2027-02-01,Overdue
1,B,2026-02-02,2027-01-01,2027-05-02,Overdue
2,C,2026-06-03,2027-03-03,2027-09-03,Sample Now
3,D,2026-08-04,2027-05-04,2027-11-04,Sample Now
4,E,2026-02-01,2027-01-01,2027-05-01,Overdue
5,F,2026-02-02,2027-01-01,2027-05-02,Overdue
6,G,2025-03-04,2027-01-01,2027-01-01,Overdue
7,H,2026-05-28,2027-02-28,2027-08-28,Sample Now
8,I,2026-05-05,2027-02-05,2027-08-05,Sample Now
9,J,2026-07-05,2027-04-05,2027-10-05,Sample Now


In [18]:
priorityList = supplyDf.sort_values(by = ["effectiveDeadline", "latestDate", "supplyRisk"], ascending=[True, True, True]).copy()
priorityList["priorityRank"] = range(1, len(priorityList) + 1)
display(priorityList[["supplyName", "supplyRisk", "priorityRank", "earliestDate","latestDate","effectiveDeadline", "samplingStatus"]])

,supplyName,supplyRisk,priorityRank,earliestDate,latestDate,effectiveDeadline,samplingStatus
6,G,1,1,2027-01-01,2027-01-01,2026-10-18,Overdue
13,N,1,2,2027-01-01,2027-01-01,2026-10-18,Overdue
18,S,1,3,2027-01-01,2027-01-01,2026-10-18,Overdue
22,W,1,4,2027-01-01,2027-01-01,2026-10-18,Overdue
25,Z,1,5,2027-01-01,2027-01-01,2026-10-18,Overdue
0,A,1,6,2027-01-01,2027-02-01,2026-11-18,Overdue
1,B,2,7,2027-01-01,2027-05-02,2027-03-13,Overdue
5,F,2,8,2027-01-01,2027-05-02,2027-03-13,Overdue
4,E,3,9,2027-01-01,2027-05-01,2027-03-22,Overdue
16,Q,4,10,2027-01-01,2027-05-21,2027-05-01,Overdue


In [37]:
coordinates = priorityList [["supplyEastings","supplyNorthings"]].to_numpy()
distanceMatrix = cdist(coordinates, coordinates) / 1000

In [38]:
distanceDf = pd.DataFrame(distanceMatrix,index = priorityList["supplyName"], columns = priorityList["supplyName"])
display(distanceDf.round(2).head(10))

supplyName,G,N,S,W,Z,A,B,F,E,Q,...,Y,U,T,V,J,D,X,R,L,K
supplyName,,,,,,,,,,,,,,,,,,,,,
G,0.00,52.45,44.95,60.29,66.20,1.43,1.55,31.96,62.67,38.74,...,52.14,47.52,48.32,54.29,47.75,6.66,13.23,45.88,69.17,33.83
N,52.45,0.00,9.80,11.59,19.55,51.04,50.94,31.02,21.01,13.71,...,12.56,21.17,10.57,37.41,8.32,56.30,44.66,28.05,21.69,34.41
S,44.95,9.80,0.00,15.38,22.03,43.59,43.51,29.28,20.42,8.16,...,8.96,13.39,15.40,29.99,12.67,48.14,38.96,19.54,24.86,24.63
W,60.29,11.59,15.38,0.00,7.96,58.95,58.86,42.20,10.68,22.72,...,9.55,19.00,22.08,32.36,19.91,63.26,54.13,25.87,10.21,36.06
Z,66.20,19.55,22.03,7.96,0.00,64.91,64.84,49.96,6.52,29.89,...,14.07,21.21,30.03,30.83,27.88,68.65,60.98,27.11,3.00,39.04
A,1.43,51.04,43.59,58.95,64.91,0.00,0.14,30.63,61.43,37.34,...,50.84,46.35,46.89,53.43,46.32,7.72,12.11,44.86,67.87,32.96
B,1.55,50.94,43.51,58.86,64.84,0.14,0.00,30.49,61.37,37.24,...,50.77,46.30,46.77,53.42,46.20,7.86,11.97,44.83,67.80,32.95
F,31.96,31.02,29.28,42.20,49.96,30.63,30.49,0.00,49.53,21.23,...,38.21,40.23,22.18,54.90,23.22,38.11,19.46,43.61,52.41,39.27
E,62.67,21.01,20.42,10.68,6.52,61.43,61.37,49.53,0.00,28.58,...,11.50,16.00,31.49,24.31,29.01,64.61,58.61,21.20,8.87,33.68


In [35]:
nearestSupplies = {}

for supply in distanceDf.index:
    distances = distanceDf.loc[supply]
    distances = distances.drop(supply)
    nearest = distances.nsmallest(5)
    nearestSupplies[supply] = nearest
    print("Supply: ", supply)
    print (nearest)

Supply:  G
supplyName
A     1431.127178
B     1551.361015
C     3308.411401
D     6656.119064
X    13231.164159
Name: G, dtype: float64
Supply:  N
supplyName
J     8324.054060
S     9804.521508
T    10567.423338
W    11590.236236
Y    12563.977157
Name: N, dtype: float64
Supply:  S
supplyName
Q     8162.726260
Y     8963.598608
N     9804.521508
M    12416.060607
J    12670.014444
Name: S, dtype: float64
Supply:  W
supplyName
Z     7963.179327
Y     9551.443137
L    10210.111508
E    10675.304679
N    11590.236236
Name: W, dtype: float64
Supply:  Z
supplyName
L     3000.746907
E     6519.906824
W     7963.179327
Y    14070.686586
N    19552.286337
Name: Z, dtype: float64
Supply:  A
supplyName
B      140.698259
G     1431.127178
C     2008.173548
D     7717.188348
X    12109.495448
Name: A, dtype: float64
Supply:  B
supplyName
A      140.698259
G     1551.361015
C     1867.481995
D     7857.197465
X    11970.089557
Name: B, dtype: float64
Supply:  F
supplyName
X    19459.690131
Q    212

In [ ]:
scheduling = priorityList.copy()
schedule = []
officerWorkload = []

for i in range (1, samplers +1):
    officerWorkload [i] = 0

while len(schedule) < len(scheduleDf):
    daily = 0
    for officer in range (1, samplers +1):
        if daily >= maxDailySamples:
            break
        if len(schedule) > len(scheduleDf):
            break
                          